In [0]:

# 1. Read matches.csv from your volume — as-is, no changes
# 2. Read deliveries.csv from your volume — as-is, no changes  
# 3. Add 3 metadata columns to both
#    — ingested_at (current timestamp)
#    — source_file (filename)
#    — ingestion_id (unique run id)
# 4. Write both as Delta tables into your catalog
#    — ipl_analytics_lakehouse.bronze.matches
#    — ipl_analytics_lakehouse.bronze.deliveries
# 5. Log record counts to a pipeline_audit table


In [0]:
#Unique ID creation [ Ingestion _id]
from datetime import datetime
ingestion_id = datetime.now().strftime("run_%Y%m%d_%H%M%S")
print(ingestion_id)


In [0]:
#1. Read matches.csv from your volume — as-is, no changes

matches_df = spark.read.csv("/Volumes/ipl_catalog/default/landing_zone_bronze_data/RAW_ZONE_BRONZE/matches.csv",
                          header=True,inferSchema=True)
matches_df.display()

In [0]:
# 2. Read deliveries.csv from your volume — as-is, no changes  
deliveries_df = spark.read.csv("/Volumes/ipl_catalog/default/landing_zone_bronze_data/RAW_ZONE_BRONZE/deliveries.csv",
                               header=True,inferSchema=True)
deliveries_df.display()

In [0]:
# 3. Add 3 metadata columns to both
#    — ingested_at (current timestamp)
#    — source_file (filename)
#    — ingestion_id (unique run id)

# In simple words:
# ColumnAnswers the questionsource_file
# Where did this row come from?
# ingested_at
# When did this row arrive?
# ingestion_id
# Which pipeline run brought this row in?


# source_file = which warehouse sent it
# ingested_at = what time it was delivered
# ingestion_id = the batch shipment number it came with

from pyspark.sql import functions as F
matches_df = matches_df.withColumn("ingested_at",F.current_timestamp())\
                        .withColumn('source_name',F.lit('matches.csv'))\
                        .withColumn('ingestion_id',F.lit(ingestion_id))

deliveries_df = deliveries_df.withColumn("ingested_at",F.current_timestamp())\
                        .withColumn('source_name',F.lit('deliveries.csv'))\
                        .withColumn('ingestion_id',F.lit(ingestion_id))

In [0]:
matches_df.display()

In [0]:
deliveries_df.display()

Notice — same ingestion_id on both tables.
That means later you can query:
sqlSELECT * FROM bronze.matches 
WHERE ingestion_id = 'run_20240115_143210'
-- and cross check with --
SELECT * FROM bronze.deliveries 
WHERE ingestion_id = 'run_20240115_143210'
Both came from the same pipeline run. Perfect traceability.

In [0]:
# 4. Write both as Delta tables into your catalog
#    — ipl_analytics_lakehouse.bronze.matches
#    — ipl_analytics_lakehouse.bronze.deliveries

matches_df.write \
          .format('delta')\
          .mode('overwrite')\
          .saveAsTable('ipl_catalog.bronze.matches')

In [0]:
deliveries_df.write \
             .format('delta')\
             .mode('overwrite')\
             .saveAsTable('ipl_catalog.bronze.deliveries')

In [0]:
matches_count = spark.table("ipl_catalog.bronze.matches").count()
deliveries_count = spark.table("ipl_catalog.bronze.deliveries").count()

print(f"matches row count     : {matches_count}")
print(f"deliveries row count  : {deliveries_count}")

In [0]:
spark.table("ipl_catalog.bronze.matches") \
    .select('ingested_at', 'source_name', 'ingestion_id') \
    .show(3, truncate=False)

In [0]:

# What Delta gives you automatically — no extra code:
# What                     HOW
# Who wrote the table      transaction log
# When it was written      transaction log
# How many files changed   transaction log
# What operation ran       WRITE, MERGE, DELETE
# Row counts per operation operationMetrics

# So your pipeline_audit table is actually bonus on top of what Delta already does.
# But here is why you still keep the audit table:

# Delta history is technical — file level, partition level, hard to read
# Your audit table is business friendly — ingestion_id, table name, record count, status in plain columns
# Your audit table works across all layers — Bronze, Silver, Gold in one place
# Delta history is per table — your audit table is the single view across your entire pipeline

In [0]:
from datetime import datetime

# Audit records for this run
audit_data = [
    (ingestion_id,
     'ipl_catalog.bronze.matches',
     spark.table("ipl_catalog.bronze.matches").count(),
     'SUCCESS',
     datetime.now()),
    (ingestion_id,
     'ipl_catalog.bronze.deliveries',
     spark.table("ipl_catalog.bronze.deliveries").count(),
     'SUCCESS',
     datetime.now())
]

# Create dataframe
audit_df = spark.createDataFrame(
    audit_data,
    ['ingestion_id', 'table_name',
     'record_count', 'status',
     'audit_timestamp']
)

# Write - always append, never overwrite
audit_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("ipl_catalog.bronze.pipeline_audit")

print("Audit log written")
audit_df.show(truncate=False)

In [0]:
spark.table("ipl_catalog.bronze.pipeline_audit").show()

In [0]:
spark.table("ipl_catalog.bronze.matches").printSchema()

In [0]:
spark.table('ipl_catalog.bronze.deliveries').printSchema()